# Week 2 results: can a regime forecast improve the strategies?

1. Train a classifier to predict whether **next month** will be stressed (SPY volatility above
   the 80th percentile of its own past year), using only information available at the time.
2. Evaluate it honestly: chronological folds with a 21-day gap, compared with two baselines,
   "always calm" and a simple "volatility is already high" rule.
3. Use its walk-forward probabilities to switch strategies into minimum variance when stress is
   forecast, and compare with the static strategies and the rule.

**Not investment advice.** Numbers in the text are for data up to 2026-09-30 (the fixed
end date in `config/universe.yaml`); re-running reproduces them.

### Key results
- **The model ranks stress modestly well:** ROC AUC 0.72 out of sample
  (2009–2026), where 0.50 is a coin flip.
- **Its probabilities are now honest:** average predicted P(stressed) 28.86% vs
  21.51% actual. An earlier version used class weighting and averaged 47.41%.
- **A one-line rule does at least as well at deciding:** at the 50.00% threshold the model's
  precision / recall / accuracy are 47.89% / 40.45% / 77.73%;
  the "vol already high" rule scores 58.33% / 51.35% /
  81.65%. Always-calm accuracy is 78.49%.
- **De-risking on the forecast cut losses in COVID but cost return overall:** equal weight + regime
  had a -16.06% COVID drawdown vs -20.32%
  static, but in 2022 it switched in and out month after month and did worse than static, and its
  Sharpe fell from 0.49 to
  0.39. Max Sharpe is the exception where the model beat the
  rule (0.50 vs 0.39),
  though both were below static Max Sharpe (0.59).

In [ ]:
import logging

import pandas as pd
from pandas.io.formats.style import Styler

from gpl import backtest as bt
from gpl import regime as rg
from gpl.data import clean_prices, get_prices, get_risk_free, load_universe
from gpl.metrics import drawdown_series, summary_table
from gpl.plots import drawdown_chart, equity_chart, regime_chart

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

cfg = load_universe()
d, R = cfg["defaults"], cfg["regime"]


def pct_table(frame: pd.DataFrame, plain: tuple[str, ...] = ()) -> Styler:
    """Fractions as 2-decimal percentages; ``plain`` columns as numbers (Brier: 3 decimals)."""
    numeric = frame.select_dtypes("number").columns
    formats = {
        c: "{:.3f}" if c == "brier" else "{:.2f}" if c in plain else "{:.2%}" for c in numeric
    }
    return frame.style.format(formats, na_rep="–")

## 1. Regime data (SPY, AGG, VIX from 2005)

The regime model uses a much longer history than the GCC ETFs (which start in Sep 2015), so it can
learn from 2008 and later crises: data from 2005-01-03, with complete features from 2006
once the one-year windows fill. Over all labelled days, 23.41% were stressed, and the
share swings a lot by year. The bar is the 80th percentile of the *previous* year's volatility, so a
turbulent year that follows a calm one clears it on many days: 2007 (48.21%),
2018 (55.78%, after 5.18% in 2017) and 2022
(55.38%, after 12.30% in 2021).

In [ ]:
regime_px = clean_prices(get_prices([R["equity"], R["bond"]], R["start"], d["end"]))
vix = get_prices(["^VIX"], R["start"], d["end"])["^VIX"]
features = rg.make_features(regime_px, vix, equity=R["equity"], bond=R["bond"])
labels = rg.make_labels(
    regime_px[R["equity"]].pct_change().iloc[1:], R["horizon"], R["quantile"], R["window"]
)
vol_rule = rg.naive_stress_signal(features, R["quantile"], R["window"])
print(f"Sample: {regime_px.index[0].date()} to {regime_px.index[-1].date()}")
print(f"Share of labelled days that were stressed: {labels.mean():.2%}")
print("By year:")
print(labels.groupby(labels.index.year).mean().dropna().map("{:.2%}".format).to_string())
features.describe().T

## 2. Out-of-sample evaluation (5 chronological folds, 21-day gap)

In [ ]:
reports = {
    m: rg.evaluate_model(
        features, labels, model=m, horizon=R["horizon"], threshold=R["threshold"], rule=vol_rule
    )
    for m in ["logistic", "random_forest"]
}
overall = pd.concat({m: r["overall"] for m, r in reports.items()})
overall = overall.drop(index=[("random_forest", "vol_rule"), ("random_forest", "always_calm")])
base = overall.loc[("logistic", "model"), "stressed_share"]
print(f"Brier score of always predicting the base rate: {base * (1 - base):.3f}")
pct_table(
    overall[
        [
            "auc",
            "precision",
            "recall",
            "accuracy",
            "flagged_share",
            "mean_p",
            "stressed_share",
            "brier",
        ]
    ],
    plain=("auc", "brier"),
)

**Reading this.** ROC AUC measures how well the model *ranks* stressed days above calm ones,
whatever threshold you pick (0.50 = coin flip, 1.00 = perfect). Precision is how often a "stressed"
call was right; compare it with the actual stressed share (21.51% in these test
folds), which is what random guessing would score. Recall is the share of stressed days flagged.
The Brier score is the mean squared gap between predicted probability and outcome; always predicting
the base rate scores 0.169.

- **Logistic:** AUC 0.72, Brier 0.163, precision 47.89%
  (+26.38 pp over the base rate), recall 40.45%.
- **Random forest:** AUC 0.68, Brier 0.161. Slightly
  better calibrated, worse at ranking, so logistic stays the default.
- **Vol rule:** AUC 0.71 (a 0/1 rule's AUC is its balanced accuracy), precision
  58.33%, recall 51.35%, accuracy 81.65%. It beats the
  model on all three at the 50.00% threshold. Volatility persists, so "it is already volatile" is a
  strong forecast of "it will be volatile".
- **Always calm:** accuracy 78.49%, higher than the model's 77.73%,
  while catching no stressed days at all, which is why accuracy is not the lead metric.

**Calibration.** The first version weighted the rare stressed class more heavily. That raised recall
but pushed the average predicted probability to 47.41% (vs 21.51% actual), so "P = 50%" didn't mean
"more likely than not". Removing the weighting fixed it: 28.86% predicted vs
21.51% actual. Time-series calibration (`CalibratedClassifierCV`) was also tried; it
ranked worse (AUC 0.67) and lost to the base rate on Brier, so it was dropped.

In [ ]:
for m, r in reports.items():
    print(m)
    display(
        pct_table(
            r["folds"][
                [
                    "test_start",
                    "test_end",
                    "stressed_share",
                    "mean_p",
                    "auc",
                    "precision",
                    "recall",
                    "accuracy",
                    "rule_precision",
                    "rule_recall",
                    "rule_accuracy",
                    "baseline_accuracy",
                ]
            ],
            plain=("auc",),
        )
    )

**By fold.** Most of the gap between predicted and actual stress comes from the first fold
(2009-07 onwards). Its model was trained only on 2006–2009, dominated by the 2008
crisis, so it expected 42.31% stress vs 15.78% actual. In the four later
folds the average prediction is within
0.94–4.08 pp
of reality. AUC varies by fold from 0.63 to
0.88: few independent episodes means noisy estimates.

## 3. Walk-forward probabilities (what an investor would have seen live)

Each month-end the model is refit on labels already known (up to 21 days earlier) and used for the
following month. The first prediction is on 2008-03-31. Over the predicted period the average
probability is 24.51% vs 21.64% actual. The chart shows month-end values,
the dates the strategies rebalance; grey months turned out stressed.

In [ ]:
p_stressed = rg.walk_forward_probabilities(
    features,
    labels,
    model=R["model"],
    refit=R["refit"],
    min_train=R["min_train"],
    horizon=R["horizon"],
)
live = pd.concat([p_stressed, labels.reindex(p_stressed.index)], axis=1).dropna()
print(f"First prediction: {p_stressed.first_valid_index().date()}")
print(f"Average P(stressed) {live.iloc[:, 0].mean():.2%} vs actual {live.iloc[:, 1].mean():.2%}")
monthly = p_stressed.resample("ME").last().dropna()
realised = labels.reindex(p_stressed.index).resample("ME").last()
regime_chart(
    monthly, R["threshold"], realised, title="Month-end probability of a stressed month ahead"
).show()

## 4. Regime-aware strategies vs static ones (2016-09 to 2026-09, 10 bps costs)

Each "+ regime" strategy switches to minimum variance at a month-end rebalance when the latest
P(stressed) is at least 50.00%; "+ vol rule" switches when the rule fires. With honest probabilities
the model de-risks in 10.83% of months and the rule in 23.33%.

In [ ]:
prices = clean_prices(get_prices(cfg["assets"], d["start"], d["end"]), max_ffill=d["max_ffill"])
rf = get_risk_free(d["start"], d["end"])
mw = d["max_weight"]
static = {
    "Equal weight": bt.equal_weight,
    "Max Sharpe": bt.max_sharpe_strategy(rf=rf, max_weight=mw),
    "Risk parity": bt.risk_parity_strategy(max_weight=mw),
}
defensive = bt.min_variance_strategy(max_weight=mw)
strategies = {
    **static,
    **{
        f"{name} + regime": bt.regime_switch_strategy(s, defensive, p_stressed, R["threshold"])
        for name, s in static.items()
    },
    **{
        f"{name} + vol rule": bt.regime_switch_strategy(s, defensive, vol_rule, 0.5)
        for name, s in static.items()
    },
    "Min variance": defensive,
    "60/40 (SPY/AGG)": bt.fixed_weights(cfg["benchmarks"]["sixty_forty"]),
    "SPY": bt.fixed_weights(cfg["benchmarks"]["spy"]),
}
run = dict(rebalance=d["rebalance"], lookback=d["lookback"], cost_bps=d["cost_bps"])
results = {name: bt.run_backtest(prices, s, **run) for name, s in strategies.items()}
returns = pd.DataFrame({n: r.returns for n, r in results.items()})

rebalance_dates = results["Equal weight"].weights.index
known = p_stressed.reindex(rebalance_dates, method="ffill")
print(
    f"Months de-risked by the model: {(known >= R['threshold']).mean():.2%}; "
    f"by the vol rule: {(vol_rule.reindex(rebalance_dates, method='ffill') >= 0.5).mean():.2%}"
)

table = summary_table(returns, rf=rf, level=d["var_level"])
table["annual_turnover"] = pd.Series(
    {n: r.turnover.iloc[1:].mean() * 12 for n, r in results.items()}
)
pct_table(
    table[
        [
            "cagr",
            "ann_vol",
            "sharpe",
            "sortino",
            "max_drawdown",
            "expected_shortfall",
            "annual_turnover",
        ]
    ].sort_values("sharpe", ascending=False),
    plain=("sharpe", "sortino", "annual_turnover"),
)

**Reading this.**
- **Every regime-aware version has a lower Sharpe than its static strategy:** equal weight
  0.49 → 0.39 (model) /
  0.44 (rule); risk parity 0.38 →
  0.30 / 0.34; max Sharpe
  0.59 → 0.50 /
  0.39.
- **The rule beats the model for equal weight and risk parity; the model beats the rule for max
  Sharpe.** A plausible reason (not tested here): the model de-risks less often, and max Sharpe is
  the strategy that gives up most by sitting in min variance.
- **Switching is expensive:** equal weight + regime trades 2.13×
  its value a year vs 0.30× static, at 10 bps on every unit
  traded.
- **Benchmarks still lead:** SPY (0.75) and 60/40 (0.67)
  top the table in this decade of strong US equities.

In [ ]:
show = [
    "Equal weight",
    "Equal weight + regime",
    "Equal weight + vol rule",
    "Risk parity",
    "Risk parity + regime",
    "Min variance",
    "60/40 (SPY/AGG)",
    "SPY",
]
equity_chart(
    pd.DataFrame({n: results[n].equity for n in show}), title="Growth of $1: static vs regime-aware"
).show()
drawdown_chart(drawdown_series(returns[show])).show()

In [ ]:
# Drawdowns in the two big stress episodes of the backtest window.
episodes = {
    "COVID (Feb-Apr 2020)": ("2020-02-01", "2020-04-30"),
    "2022 rates shock": ("2022-01-01", "2022-10-31"),
}
pct_table(
    pd.DataFrame(
        {ep: drawdown_series(returns.loc[a:b]).min() for ep, (a, b) in episodes.items()}
    ).loc[show]
)

In [ ]:
# 2022 in detail: month-end forecasts and calendar-year results.
print("Month-end P(stressed), Jan-Jul 2022:")
print(p_stressed.resample("ME").last().loc["2022-01":"2022-07"].map("{:.2%}".format).to_string())
print(f"Share of 2022 days labelled stressed: {labels.loc['2022'].mean():.2%}")
agg_2022 = prices["AGG"].loc["2022"].iloc[-1] / prices["AGG"].loc["2021"].iloc[-1] - 1
print(f"AGG in 2022: {agg_2022:.2%}")
pct_table(
    pd.DataFrame(
        {
            name: {
                "2022 return": (1 + results[name].returns.loc["2022"]).prod() - 1,
                "2022 costs": results[name].costs.loc["2022"].sum(),
            }
            for name in ["Equal weight", "Equal weight + regime", "Min variance"]
        }
    ).T
)

**Stress episodes.** In COVID, de-risking helped: equal weight fell -20.32%
static vs -16.06% with the model (the same as with the rule) and
-14.18% for min variance. In the 2022 rates shock it did not:
equal weight + regime fell -18.24% vs
-17.95% static; the rule did slightly better
(-16.49%).

2022 *was* clearly stressed (55.38% of its days labelled stressed), so the problem was not
detection but whipsaw. The model's month-end probability crossed 50.00% back and forth: January
58.11%, February 39.76%, March
24.08%, April 71.65%, May
24.36%, June 67.50%, July
10.53%. The strategy switched in and out of min variance, paying
0.44% in costs that year vs 0.04% static,
and ended 2022 at -13.18% vs -12.57%. Even
holding min variance throughout would only have helped modestly
(-11.52%), because bonds fell too (AGG -13.02%).

## Limitations

- **One forecast target:** stress is defined by SPY volatility. GCC-specific shocks (oil, local
  politics) that don't move SPY are invisible to this model.
- **Few independent episodes:** 20 years of daily rows sounds like a lot, but there are only a
  handful of separate stress periods, and overlapping 21-day labels make neighbouring rows nearly
  identical. Fold AUCs range from 0.63 to 0.88.
- **Threshold choice:** with calibrated probabilities, 50.00% means "more likely than not". A lower
  threshold would trade precision for recall; choosing it on this same sample would be overfitting.
- **Base-rate drift:** probabilities are calibrated to the training period's stress rate. After a
  crisis-heavy training window (the first fold) the model over-predicts stress.
- **Whipsaw:** a month-by-month switch reacts to a probability that can jump across the threshold
  and back, as in 2022. Requiring two months above the threshold, or scaling risk gradually with
  P(stressed), are untested alternatives.
- **Switching to min variance** is one of many possible de-risking rules, and it is bond-heavy, which
  helped only modestly when bonds and stocks fell together in 2022.